# 3.1 Hardware testing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import display

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from style import (
    PRIMARY_COLOR,
    SECONDARY_COLOR,
    DARK_TEXT,
    HEATMAP_CMAP,
    finish_axis,
    save_figure,
)

### 3.1.1 Frequency-response characterisation of the final patch version

Load the channel-by-frequency SNR diagnostics for the final silicone-coated patch (ten excitation frequencies, Channels 1-8, MIC and ACC).

In [ ]:
FREQUENCY_RESPONSE_DIR = (
    Path.home() / "Desktop" / "QMUL BME" / "TFM" / "Freq new patch" / "20260603"
)

ACC_DIAG_FILE = (
    FREQUENCY_RESPONSE_DIR
    / "FFT ACC"
    / "quantitative_diagnostics"
    / "acc_quantitative_frequency_diagnostics_channels_1_to_8.csv"
)
MIC_DIAG_FILE = (
    FREQUENCY_RESPONSE_DIR
    / "FFT MIC"
    / "quantitative_diagnostics"
    / "mic_quantitative_frequency_diagnostics_channels_1_to_8.csv"
)

acc_raw = pd.read_csv(ACC_DIAG_FILE)
mic_raw = pd.read_csv(MIC_DIAG_FILE)

#### Figure 9 — Target-frequency SNR heatmaps (Channels 1-8, MIC and ACC)

In [ ]:
acc_data = acc_raw.groupby(["expected_frequency_Hz", "channel"], as_index=False).agg(
    target_snr_dB=("target_snr_dB", "mean")
)

mic_data = mic_raw.groupby(["expected_frequency_Hz", "channel"], as_index=False).agg(
    target_snr_dB=("target_snr_dB", "mean")
)

acc_heatmap = (
    acc_data.pivot(index="channel", columns="expected_frequency_Hz", values="target_snr_dB")
    .sort_index(ascending=False)
    .sort_index(axis=1)
)

mic_heatmap = (
    mic_data.pivot(index="channel", columns="expected_frequency_Hz", values="target_snr_dB")
    .sort_index(ascending=False)
    .sort_index(axis=1)
)

assert acc_heatmap.shape == (8, 10)
assert mic_heatmap.shape == (8, 10)

fig = plt.figure(figsize=(12.6, 4.8))

grid = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 0.045], wspace=0.24)

ax_acc = fig.add_subplot(grid[0, 0])
ax_mic = fig.add_subplot(grid[0, 1])
cax = fig.add_subplot(grid[0, 2])

vmin = 0
vmax = 60

image_acc = ax_acc.imshow(
    acc_heatmap.values,
    cmap=HEATMAP_CMAP,
    vmin=vmin,
    vmax=vmax,
    aspect="auto",
    origin="upper",
    interpolation="nearest",
)

image_mic = ax_mic.imshow(
    mic_heatmap.values,
    cmap=HEATMAP_CMAP,
    vmin=vmin,
    vmax=vmax,
    aspect="auto",
    origin="upper",
    interpolation="nearest",
)


def format_heatmap(ax, matrix, title, show_ylabels):
    x_labels = [str(int(value)) for value in matrix.columns]
    y_labels = [f"Ch {int(value)}" for value in matrix.index]

    ax.set_title(title, loc="left", fontsize=13, fontweight="semibold", pad=7)

    ax.set_xticks(np.arange(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=35, ha="right")

    ax.set_yticks(np.arange(len(y_labels)))

    if show_ylabels:
        ax.set_yticklabels(y_labels)
        ax.set_ylabel("Channel")
    else:
        ax.set_yticklabels([])
        ax.set_ylabel("")

    ax.set_xlabel("Expected excitation frequency (Hz)")

    ax.set_xticks(np.arange(-0.5, matrix.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, matrix.shape[0], 1), minor=True)

    ax.grid(which="minor", color="white", linewidth=0.7, alpha=0.5)

    ax.tick_params(which="minor", bottom=False, left=False)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = float(matrix.iloc[row_index, column_index])

            text_color = "white" if value >= 32 else DARK_TEXT

            ax.text(
                column_index,
                row_index,
                f"{value:.1f}",
                ha="center",
                va="center",
                fontsize=9.5,
                color=text_color,
            )


format_heatmap(ax_acc, acc_heatmap, "(a) Accelerometers", show_ylabels=True)

format_heatmap(ax_mic, mic_heatmap, "(b) Microphones", show_ylabels=False)

colorbar = fig.colorbar(image_mic, cax=cax)

colorbar.set_label("Target SNR (dB)", fontsize=13)

colorbar.set_ticks([0, 10, 20, 30, 40, 50, 60])

colorbar.ax.tick_params(labelsize=11)

fig.subplots_adjust(left=0.075, right=0.955, bottom=0.20, top=0.86)

save_figure(fig, "figure_12_target_snr_heatmaps")

plt.show()

#### Figure 10 — Relative frequency-response profiles after correction for the measured dBC reference

In [ ]:
def midpoint(low, high):
    return (low + high) / 2.0


dbc_reference_df = pd.DataFrame(
    {
        "expected_frequency_Hz": [10, 60, 110, 160, 210, 260, 310, 360, 410, 460],
        "MIC_sound_level_dBC": [
            midpoint(64.0, 67.2),
            midpoint(72.5, 73.0),
            85.8,
            96.0,
            95.2,
            95.2,
            97.8,
            96.9,
            95.8,
            93.8,
        ],
        "ACC_sound_level_dBC": [
            midpoint(63.8, 66.1),
            midpoint(72.5, 72.8),
            85.5,
            91.1,
            95.8,
            95.0,
            97.1,
            96.8,
            96.3,
            92.8,
        ],
    }
)

dbc_reference_df["MIC_sound_reference_linear"] = 10 ** (
    dbc_reference_df["MIC_sound_level_dBC"] / 20.0
)

dbc_reference_df["ACC_sound_reference_linear"] = 10 ** (
    dbc_reference_df["ACC_sound_level_dBC"] / 20.0
)


def clean_frequency_response_data(data, sensor_type):
    result = data.copy()
    result["sensor_type"] = sensor_type

    result["channel"] = (
        result["channel"].astype(str).str.extract(r"(\d+)", expand=False).astype(float)
    )

    result = result[result["channel"].between(1, 8)].copy()

    result["channel"] = result["channel"].astype(int)
    result["expected_frequency_Hz"] = pd.to_numeric(
        result["expected_frequency_Hz"], errors="coerce"
    )
    result["target_peak_amplitude"] = pd.to_numeric(
        result["target_peak_amplitude"], errors="coerce"
    )

    return result


mic_figure14 = clean_frequency_response_data(mic_raw, "MIC")

acc_figure14 = clean_frequency_response_data(acc_raw, "ACC")

figure14_channels = pd.concat([mic_figure14, acc_figure14], ignore_index=True)

figure14_channels = figure14_channels.groupby(
    ["sensor_type", "expected_frequency_Hz", "channel"], as_index=False
).agg(target_peak_amplitude=("target_peak_amplitude", "mean"))

figure14_channels = figure14_channels.merge(
    dbc_reference_df, on="expected_frequency_Hz", how="left", validate="many_to_one"
)

figure14_channels["sound_reference_linear_used"] = np.where(
    figure14_channels["sensor_type"].eq("MIC"),
    figure14_channels["MIC_sound_reference_linear"],
    figure14_channels["ACC_sound_reference_linear"],
)

figure14_channels["dBC_normalised_response"] = (
    figure14_channels["target_peak_amplitude"]
    / figure14_channels["sound_reference_linear_used"]
)

figure14_summary = figure14_channels.groupby(
    ["sensor_type", "expected_frequency_Hz"], as_index=False
).agg(
    n_channels=("channel", "count"),
    mean_corrected_response=("dBC_normalised_response", "mean"),
    sd_corrected_response=("dBC_normalised_response", "std"),
)

figure14_summary["sem_corrected_response"] = figure14_summary[
    "sd_corrected_response"
] / np.sqrt(figure14_summary["n_channels"])

figure14_summary["relative_response"] = np.nan
figure14_summary["relative_sem"] = np.nan

for sensor_type in ["MIC", "ACC"]:
    mask = figure14_summary["sensor_type"].eq(sensor_type)

    maximum_mean = figure14_summary.loc[mask, "mean_corrected_response"].max()

    figure14_summary.loc[mask, "relative_response"] = (
        figure14_summary.loc[mask, "mean_corrected_response"] / maximum_mean
    )

    figure14_summary.loc[mask, "relative_sem"] = (
        figure14_summary.loc[mask, "sem_corrected_response"] / maximum_mean
    )

peak_frequencies = (
    figure14_summary.loc[
        figure14_summary.groupby("sensor_type")["relative_response"].idxmax(),
        ["sensor_type", "expected_frequency_Hz", "relative_response"],
    ]
    .sort_values("sensor_type")
    .reset_index(drop=True)
)

display(peak_frequencies)

assert peak_frequencies["expected_frequency_Hz"].eq(60).all(), (
    "The reconstructed curves do not peak at 60 Hz. "
    "Check that acc_raw and mic_raw come from the correct diagnostic CSV files."
)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

for sensor_type, color, label in [
    ("MIC", PRIMARY_COLOR, "MIC mean across Channels 1–8"),
    ("ACC", SECONDARY_COLOR, "ACC mean across Channels 1–8"),
]:
    data = figure14_summary[figure14_summary["sensor_type"] == sensor_type].sort_values(
        "expected_frequency_Hz"
    )

    x = data["expected_frequency_Hz"].to_numpy()
    y = data["relative_response"].to_numpy()
    error = data["relative_sem"].to_numpy()

    ax.plot(x, y, marker="o", color=color, label=label)

    ax.fill_between(x, y - error, y + error, color=color, alpha=0.16, linewidth=0)

ax.set_title(
    "Relative frequency-response shape after external dBC correction",
    fontsize=13,
    fontweight="semibold",
    pad=7,
)

ax.set_xlabel("Excitation frequency (Hz)")

ax.set_ylabel("Relative dBC-normalised response")

ax.set_xticks(dbc_reference_df["expected_frequency_Hz"])

ax.tick_params(axis="x", rotation=35)

ax.set_ylim(0, 1.10)

ax.legend(frameon=False, loc="upper right")

finish_axis(ax)

fig.tight_layout()

save_figure(fig, "figure_14_relative_frequency_response")

plt.show()